In [ ]:
import pandas as pd
import sqlite3
import numpy as np

# 1. DATA PREP: Handling Encoding & Nulls
# -------------------------------------------------------------------------
# Encountered errors with UTF-8, so switched to ISO-8859-1 for the City/Region columns.
# Dropping rows with null shipping dates to ensure the SLA math is valid.
# -------------------------------------------------------------------------
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='ISO-8859-1')
df = df.dropna(subset=['Days for shipping (real)', 'Days for shipment (scheduled)', 'Customer Id'])

conn = sqlite3.connect(':memory:')
df.to_sql('supply_chain', conn, index=False)

print("SQL Database initialized. Ready for Audit.")

# 2. STAGE 1: INTEGRITY AUDIT (Is the data reliable?)
# -------------------------------------------------------------------------
# Checking for duplicates and financial 'noise' (like $0 orders) 
# before starting the main analysis.
# -------------------------------------------------------------------------
query_audit = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT "Order Id") AS unique_orders,
    SUM(CASE WHEN "Order Item Total" <= 0 THEN 1 ELSE 0 END) AS price_errors,
    SUM(CASE WHEN "Days for shipping (real)" > 10 THEN 1 ELSE 0 END) AS extreme_delays
FROM supply_chain;
"""
print("\n--- [STAGE 1] DATA INTEGRITY AUDIT ---")
print(pd.read_sql_query(query_audit, conn))

# 3. STAGE 2: SLA PERFORMANCE (Strict vs. Buffered)
# -------------------------------------------------------------------------
# Testing the 'Promise' accuracy. 
# Added a 1-day 'Grace Period' (Buffered) to see if failures are just 
# marginal delays or massive operational gaps.
# -------------------------------------------------------------------------
query_failure = """
SELECT 
    "Shipping Mode",
    COUNT(*) AS order_volume,
    -- Strict: Arrived on/before the exact promised date
    ROUND(AVG(CASE WHEN "Days for shipping (real)" <= "Days for shipment (scheduled)" THEN 1.0 ELSE 0.0 END) * 100, 2) AS strict_success_rate,
    -- Buffered: Arrived within 1 day of the promise
    ROUND(AVG(CASE WHEN "Days for shipping (real)" <= ("Days for shipment (scheduled)" + 1) THEN 1.0 ELSE 0.0 END) * 100, 2) AS buffered_success_rate
FROM supply_chain
GROUP BY 1
ORDER BY order_volume DESC;
"""
print("\n--- [STAGE 2] SLA PERFORMANCE: RELIABILITY ---")
print(pd.read_sql_query(query_failure, conn))

# 4. STAGE 3: FULFILLMENT FUNNEL (Latency Analysis)
# -------------------------------------------------------------------------
# Measuring 'Latency'—the actual gap in days between promise and delivery.
# This helps pinpoint which shipping tier is 'leaking' the most time.
# -------------------------------------------------------------------------
query_funnel = """
SELECT 
    "Shipping Mode",
    AVG("Days for shipment (scheduled)") AS promised_days,
    AVG("Days for shipping (real)") AS actual_days,
    -- Latency Gap: Average delay per order in days.
    ROUND(AVG("Days for shipping (real)" - "Days for shipment (scheduled)"), 2) AS avg_latency_gap
FROM supply_chain
GROUP BY 1
ORDER BY avg_latency_gap DESC;
"""
print("\n--- [STAGE 3] FULFILLMENT FUNNEL: LATENCY GAP ---")
print(pd.read_sql_query(query_funnel, conn))

# 5. STAGE 4: USER SEGMENTATION (LTV Impact)
# -------------------------------------------------------------------------
# Using a CTE to identify if logistics failures are hitting our 
# high-value 'Priority' customers.
# -------------------------------------------------------------------------
query_segmentation = """
WITH UserValue AS (
    SELECT 
        "Customer Id",
        SUM("Order Item Total") AS total_spent,
        AVG(Late_delivery_risk) AS delay_rate
    FROM supply_chain
    GROUP BY 1
)
SELECT 
    CASE 
        WHEN total_spent > 500 THEN 'Priority (High Spend)'
        WHEN total_spent BETWEEN 200 AND 500 THEN 'Standard (Mid Spend)'
        ELSE 'Casual (Low Spend)'
    END AS customer_segment,
    COUNT(*) AS user_count,
    ROUND(AVG(delay_rate) * 100, 2) AS avg_failure_pct
FROM UserValue
GROUP BY 1
ORDER BY avg_failure_pct DESC;
"""
print("\n--- [STAGE 4] SEGMENTATION: IMPACT ON PRIORITY USERS ---")
print(pd.read_sql_query(query_segmentation, conn))

# 6. STAGE 5: SIMULATED A/B TEST (Expected Delivery Dates)
# -------------------------------------------------------------------------
# Hypothesis: Shifting the promise from 1-day to 4-days will align 
# expectations with our actual network speed without costing a dollar.
# -------------------------------------------------------------------------
query_ab_test = """
SELECT 
    CASE WHEN ("Order Id" % 2 = 0) THEN 'Control (1-Day Promise)' ELSE 'Variant (4-Day Estimate)' END AS test_group,
    ROUND(AVG(CASE 
        WHEN ("Order Id" % 2 != 0 AND "Days for shipping (real)" <= 4) THEN 1.0 
        WHEN ("Order Id" % 2 = 0 AND "Days for shipping (real)" <= "Days for shipment (scheduled)") THEN 1.0
        ELSE 0.0 
    END) * 100, 2) AS fulfillment_success_rate
FROM supply_chain
WHERE "Shipping Mode" = 'First Class'
GROUP BY 1;
"""
print("\n--- [STAGE 5] A/B TEST: SERVICE LEVEL RECOVERY ---")
print(pd.read_sql_query(query_ab_test, conn))